In [1]:
import pandas as pd
import numpy as np
import gc

In [2]:
df_Master = pd.read_parquet('df_Master_Final.parquet')

In [3]:
gc.collect()

0

In [4]:
df_Master['hour'] = df_Master['created_at'].dt.hour
gc.collect()
df_Master['month'] = df_Master['created_at'].dt.month
gc.collect()
df_Master['day_name'] = df_Master['created_at'].dt.day_name()
gc.collect()
df_Master['is_weekend'] = np.where(
    df_Master['day_name'].isin(['Saturday', 'Sunday']),
    'Weekend',
    'Weekday'
)
gc.collect()
df_Master['member_status'] = np.where(df_Master['user_id'].notna(), 'Member', 'Guest')
gc.collect()
df_Master['is_voucher_used'] = np.where(
    df_Master['voucher_id'].notna(),
    'Voucher',
    'No Voucher'
)
gc.collect()
df_Master['month_name'] = (
    df_Master['created_at']
    .dt.month_name()
)
gc.collect()
df_Master['transaction_period'] = np.select(
    [
        df_Master['hour'].between(5, 10),
        df_Master['hour'].between(11, 15),
        df_Master['hour'].between(16, 19),
        df_Master['hour'].between(20, 23)
    ],
    [
        'Morning',
        'Afternoon',
        'Evening',
        'Night'
    ],
    default='Late Night'
)
df_Master['discount_ratio'] = (
    df_Master['discount_applied'] / (df_Master['original_amount'] + 1e-6)
)
gc.collect()

0

In [5]:
print(df_Master.columns)

Index(['transaction_id', 'item_id', 'quantity', 'unit_price', 'subtotal',
       'item_name', 'category_x', 'price', 'is_seasonal', 'store_id',
       'payment_method_id', 'voucher_id', 'user_id', 'original_amount',
       'discount_applied', 'final_amount', 'Non-Member', 'Member',
       'voucher_code', 'discount_type', 'discount_value', 'valid_from',
       'valid_to', 'calculated_discount', 'gender', 'birthdate',
       'registered_at', 'store_name', 'street', 'postal_code', 'city', 'state',
       'latitude', 'longitude', 'created_at', 'method_id', 'method_name',
       'category_y', 'hour', 'month', 'day_name', 'is_weekend',
       'member_status', 'is_voucher_used', 'month_name', 'transaction_period',
       'discount_ratio'],
      dtype='object')


In [6]:
gc.collect()

0

In [7]:
df_Master = df_Master.rename(columns={
    'category_x': 'menu_category',
    'category_y': 'payment_category'
})
gc.collect()


24

In [8]:
df_Master = df_Master.drop(
    columns=['method_id', 'Non-Member', 'Member'],
    errors='ignore'
)
gc.collect()

0

In [14]:
df_transaction_features = (
    df_Master
    .groupby('transaction_id')
    .agg({
        'quantity': 'sum',
        'final_amount': 'first',
        'discount_applied': 'first',
        'is_weekend': 'first',
        'is_voucher_used': 'first',
        'hour': 'first',
        'month_name': 'first',
        'day_name': 'first',
        'city': 'first',
        'method_name': 'first',
        'payment_category': 'first',
        'member_status': 'first',
        'created_at': 'first',
        'user_id': 'first',
        'transaction_period': 'first'
    })
    .reset_index()
    .rename(columns={
        'quantity': 'basket_size'
    })
)
gc.collect()

0

In [10]:
snapshot_date = (
    df_transaction_features['created_at'].max()
    + pd.Timedelta(days=1)
)
gc.collect()

rfm_table = (
    df_transaction_features[
        df_transaction_features['user_id'].notna()
    ]
    .groupby('user_id')
    .agg(
        Recency=('created_at', lambda x: (snapshot_date - x.max()).days),
        Frequency=('transaction_id', 'count'),
        Monetary=('final_amount', 'sum')
    )
    .reset_index()
)
gc.collect()
rfm_table['is_repeat_customer'] = np.where(
    rfm_table['Frequency'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)
gc.collect()

0

In [ ]:
df_Master.to_parquet(
    'df_Master_FE.parquet',
    index=False
)
gc.collect()
df_transaction_features.to_parquet(
    'df_transaction_features.parquet',
    index=False
)
gc.collect()
rfm_table.to_parquet(
    'df_rfm.parquet',
    index=False
)

In [13]:
gc.collect()

0